<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/03d_pubmedbert_200k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03d – PubMedBERT trained on PubMed 200k RCT (filtered) – additional experiment
Same model and input as the best 20k model (sentence + position + neighbours), trained on the
filtered 200k training set from 02b (validation and test abstracts removed). Evaluated on the SAME
PubMed 20k RCT validation set, so it can be compared directly with 03c.
This is an additional experiment on training-set size, not part of the main model comparison.

In [7]:
# ===== CONFIGURATION =====
USE_CONTEXT = True         # same input as our best 20k model (sentence + position + neighbours)
SUBSAMPLE = None          # quick test first; then None = all ~2.15 million training sentences
SEED = 42

PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
TRAIN_FILE = f'{DATA_DIR}/PubMed_200k_RCT_train_filtered.csv'   # from 02b (dev/test abstracts removed)
DEV_FILE = f'{DATA_DIR}/PubMed_20k_RCT_dev.csv'                  # the SAME validation set as always
TEST_FILE = f'{DATA_DIR}/PubMed_20k_RCT_test.csv'                # only read to check for leakage
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']

EPOCHS = 1                 # one pass over 12x more data
LEARNING_RATE = 3e-5
BATCH_SIZE, MAX_LENGTH = 32, 256
SAVE_EVERY_STEPS = 10000   # safety copy of the model during training (about every 320,000 sentences)

RUN_NAME = 'pubmedbert_context_200k_filtered' + (f'_sub{SUBSAMPLE}' if SUBSAMPLE else '')
MODEL_DIR = f'{PROJECT_DIR}/models/{RUN_NAME}'
PREDICTIONS_PATH = f'{PROJECT_DIR}/predictions/dev_{RUN_NAME}.csv'
print('Run:', RUN_NAME)

Run: pubmedbert_context_200k_filtered


In [8]:
import os
import torch
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(os.path.dirname(PREDICTIONS_PATH), exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '|', torch.cuda.get_device_name(0) if device == 'cuda' else 'NO GPU')

columns = ['abstract_id', 'sentence_idx', 'label', 'text', 'context', 'y']   # only what we need (saves memory)
train = pd.read_csv(TRAIN_FILE, dtype={'abstract_id': str}, keep_default_na=False, usecols=columns)
dev = pd.read_csv(DEV_FILE, dtype={'abstract_id': str}, keep_default_na=False, usecols=columns)
test_ids = set(pd.read_csv(TEST_FILE, dtype={'abstract_id': str}, keep_default_na=False,
                           usecols=['abstract_id']).abstract_id)

# Leakage checks: no validation or test abstract may be in the training data
assert len(dev) == 30212, 'Unexpected validation size: re-run 02_data_preparation'
assert not set(train.abstract_id) & set(dev.abstract_id), 'Leakage: dev abstract in training data'
assert not set(train.abstract_id) & test_ids, 'Leakage: test abstract in training data'
print(f'✅ Train: {train.abstract_id.nunique():,} abstracts / {len(train):,} sentences | '
      f'Dev: {len(dev):,} sentences | no dev/test abstract in training data')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | NVIDIA L4
✅ Train: 185,792 abstracts / 2,153,197 sentences | Dev: 30,212 sentences | no dev/test abstract in training data


In [9]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentenceDataset(Dataset):
    """Keeps the raw text; it is tokenised batch by batch during training (2 million sentences
    tokenised in advance would not fit in memory)."""
    def __init__(self, df):
        self.texts = df.text.tolist()
        self.contexts = df.context.tolist()
        self.labels = df.y.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.contexts[i], self.labels[i]

def collate(batch):
    """Tokenise one batch. padding='longest' pads only up to the longest sentence of the batch."""
    texts, contexts, labels = zip(*batch)
    if USE_CONTEXT:
        enc = tokenizer(list(texts), list(contexts), truncation='longest_first',
                        max_length=MAX_LENGTH, padding='longest', return_tensors='pt')
    else:
        enc = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH,
                        padding='longest', return_tensors='pt')
    return enc['input_ids'], enc['attention_mask'], torch.tensor(labels)

train_run = train.sample(SUBSAMPLE, random_state=SEED) if SUBSAMPLE else train
train_loader = DataLoader(SentenceDataset(train_run), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=2,
                          generator=torch.Generator().manual_seed(SEED))
dev_loader = DataLoader(SentenceDataset(dev), batch_size=256, shuffle=False,
                        collate_fn=collate, num_workers=2)
print(f'Training sentences: {len(train_run):,} | batches per epoch: {len(train_loader):,}')

Training sentences: 2,153,197 | batches per epoch: 67,288


In [10]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

torch.manual_seed(SEED)
np.random.seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES)).to(device)
optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimiser, int(0.06 * total_steps), total_steps)
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))

@torch.no_grad()
def predict(loader):
    """Return the probability of each class for every sentence (n x 5)."""
    model.eval()
    probs = []
    for ids, mask, _ in loader:
        with torch.autocast(device, enabled=(device == 'cuda')):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(probs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ide

In [13]:
import time, json
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

best_f1, start, log, step = -1, time.time(), [], 0

for epoch in range(EPOCHS):
    model.train()
    for ids, mask, y_batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
        optimiser.zero_grad()
        with torch.autocast(device, enabled=(device == 'cuda')):
            loss = model(input_ids=ids.to(device), attention_mask=mask.to(device),
                         labels=y_batch.to(device)).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimiser)
        scaler.update()
        scheduler.step()
        step += 1

        # Safety copy: if Colab disconnects, the work done so far is not lost
        if step % SAVE_EVERY_STEPS == 0:
            model.save_pretrained(f'{MODEL_DIR}/checkpoint')
            tokenizer.save_pretrained(f'{MODEL_DIR}/checkpoint')
            json.dump({'step': step, 'of_total_steps': total_steps,
                       'minutes': round((time.time() - start) / 60, 1)},
                      open(f'{MODEL_DIR}/checkpoint/progress.json', 'w'))

    dev_f1 = f1_score(dev.y, predict(dev_loader).argmax(1), average='macro')
    minutes = (time.time() - start) / 60
    log.append({'epoch': epoch + 1, 'dev_macro_f1': round(dev_f1, 4), 'minutes': round(minutes, 1)})
    print(f'Epoch {epoch + 1}: dev macro-F1 = {dev_f1:.4f} | {minutes:.1f} min')
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        print('  → Best so far, saved to Drive')

run_info = {'run': RUN_NAME, 'model': MODEL_NAME, 'train_file': TRAIN_FILE,
            'train_abstracts': int(train_run.abstract_id.nunique()), 'train_sentences': len(train_run),
            'use_context': USE_CONTEXT, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
            'max_length': MAX_LENGTH, 'learning_rate': LEARNING_RATE, 'seed': SEED,
            'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu',
            'total_minutes': round((time.time() - start) / 60, 1), 'log': log}
json.dump(run_info, open(f'{MODEL_DIR}/run_info.json', 'w'), indent=2)
print(f"\nTotal training time: {run_info['total_minutes']} minutes")

Epoch 1/1:   0%|          | 0/67288 [00:00<?, ?it/s]

AssertionError: No inf checks were recorded for this optimizer.

In [12]:
from sklearn.metrics import accuracy_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
dev_probs = predict(dev_loader)
dev_pred = dev_probs.argmax(1)

print('Accuracy:', round(accuracy_score(dev.y, dev_pred), 3))
print('Macro-F1:', round(f1_score(dev.y, dev_pred, average='macro'), 3))
print()
print(classification_report(dev.y, dev_pred, target_names=CLASSES, digits=3))

output = dev[['abstract_id', 'sentence_idx', 'label']].copy()
output['prediction'] = [CLASSES[i] for i in dev_pred]
for i, c in enumerate(CLASSES):
    output[f'prob_{c}'] = dev_probs[:, i]
output.to_csv(PREDICTIONS_PATH, index=False)
print('Predictions saved to', PREDICTIONS_PATH)

# Same validation set, so we can compare directly with the model trained on 20k
reference = pd.read_csv(f'{PROJECT_DIR}/predictions/dev_pubmedbert_context.csv',
                        dtype={'abstract_id': str}, keep_default_na=False)
both = output.merge(reference, on=['abstract_id', 'sentence_idx', 'label'], suffixes=('_200k', '_20k'))
assert len(both) == len(dev), 'The two prediction files do not match'
print('\nMacro-F1 on the same validation set:')
print('  trained on 20k  :', round(f1_score(both.label, both.prediction_20k, average='macro'), 4))
print('  trained on 200k :', round(f1_score(both.label, both.prediction_200k, average='macro'), 4))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Accuracy: 0.94
Macro-F1: 0.898

              precision    recall  f1-score   support

  BACKGROUND      0.791     0.863     0.826      3449
   OBJECTIVE      0.792     0.692     0.739      2376
     METHODS      0.970     0.983     0.976      9964
     RESULTS      0.978     0.970     0.974      9841
 CONCLUSIONS      0.980     0.967     0.973      4582

    accuracy                          0.940     30212
   macro avg      0.902     0.895     0.898     30212
weighted avg      0.940     0.940     0.939     30212

Predictions saved to /content/drive/MyDrive/exsu_project/predictions/dev_pubmedbert_context_200k_filtered.csv

Macro-F1 on the same validation set:
  trained on 20k  : 0.884
  trained on 200k : 0.8977
